In [ ]:
import os  # <--- Certifique-se de importar o 'os' no topo do arquivo!
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Functions.Utils import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as rmse
import torch
import random
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import math
from torch.utils.data import random_split

import numpy as np
import pandas as pd
from casadi import *
import do_mpc
from scipy.spatial import KDTree
import plotly.graph_objects as go


In [ ]:
path = r'DyntheticDataset\OuterRace.csv'
df = pd.read_csv(path)
x = df['x'].values
y = df['y'].values
e=400
PlotPLY(x[:e],y[:e])

In [ ]:
def create_track_boundaries(x, y, thickness=5.0):
    half_w = thickness / 2.0
    
    # Check if the track forms a closed loop (start point equals end point)
    is_closed = np.allclose([x[0], y[0]], [x[-1], y[-1]])
    
    if is_closed:
        # Pad coordinates periodically to ensure smooth gradients at the loop joint
        x_pad = np.concatenate(([x[-2]], x, [x[1]]))
        y_pad = np.concatenate(([y[-2]], y, [y[1]]))
        dx = np.gradient(x_pad)[1:-1]
        dy = np.gradient(y_pad)[1:-1]
    else:
        # Standard central/edge differences for open tracks
        dx = np.gradient(x)
        dy = np.gradient(y)
        
    # Perpendicular normal vectors
    normals = np.vstack((-dy, dx)).T
    norms = np.linalg.norm(normals, axis=1)
    
    # Avoid division by zero on identical consecutive points
    norms[norms == 0] = 1e-8
    normals /= norms[:, np.newaxis]
    
    # Compute inner and outer boundaries
    x_inner = x + half_w * normals[:, 0]
    y_inner = y + half_w * normals[:, 1]
    x_outer = x - half_w * normals[:, 0]
    y_outer = y - half_w * normals[:, 1]
    
    return x_inner, y_inner, x_outer, y_outer

# 2. Compute boundaries
x_inner, y_inner, x_outer, y_outer = create_track_boundaries(x, y, thickness=5.0)

PlotSeriesPLY(xSeries=[x_outer,x,x_inner],ySeries=[y_outer,y,y_inner], names=['outer','mid','inner'])

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import sys
from casadi import *
import os
rel_do_mpc_path = os.path.join('..','..','..')
sys.path.append(rel_do_mpc_path)
import do_mpc

# ==========================================
# 1. LOAD RACETRACK DATA (Midline)
# ==========================================

path = r'DyntheticDataset\OuterRace.csv'
df = pd.read_csv(path)
x_mid = df['x'].values
y_mid = df['y'].values


In [ ]:
model_type = 'continuous' # either 'discrete' or 'continuous'
model = do_mpc.model.Model(model_type)

x1  = model.set_variable(var_type='_x', var_name='x1', shape=(1,1))
x2  = model.set_variable(var_type='_x', var_name='x2', shape=(1,1))
x3  = model.set_variable(var_type='_x', var_name='x3', shape=(1,1))
x4  = model.set_variable(var_type='_x', var_name='x4', shape=(1,1))

dx = model.set_variable(var_type='_x', var_name='dx', shape=(4,1))
u1  = model.set_variable(var_type='_u', var_name='u1', shape=(1,1))
u2  = model.set_variable(var_type='_u', var_name='u2', shape=(1,1))

l = model.set_variable('parameter', 'l')

model.set_rhs('x1', dx[0])
model.set_rhs('x2', dx[1])
model.set_rhs('x3', dx[2])
model.set_rhs('x4', dx[3])

In [ ]:
dx_next = vertcat(
    x3 * np.cos(x4 + np.arctan(0.5 * np.tan(u2))),
    x3 * np.sin(x4 + np.arctan(0.5 * np.tan(u2))),
    u1,
    (x3/l) * (np.sin(np.arctan(0.5 * np.tan(u2))))
)

model.set_rhs('dx', dx_next)

model.setup()

In [ ]:
import numpy as np
import pandas as pd
from casadi import *
import do_mpc
from scipy.spatial import KDTree
import plotly.graph_objects as go

# ==========================================
# 1. LOAD RACETRACK DATA (Midline)
# ==========================================
path = r'DyntheticDataset\OuterRace.csv'
df = pd.read_csv(path)
x_mid = df['x'].values
y_mid = df['y'].values

# Create a spatial lookup tree to automatically find the closest track coordinate
track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

# ==========================================
# 2. MODEL DEFINITION
# ==========================================
model_type = 'continuous' 
model = do_mpc.model.Model(model_type)

# States
x_pos = model.set_variable(var_type='_x', var_name='x_pos') 
y_pos = model.set_variable(var_type='_x', var_name='y_pos') 
v     = model.set_variable(var_type='_x', var_name='v')     
psi   = model.set_variable(var_type='_x', var_name='psi')   

# Control Inputs
a     = model.set_variable(var_type='_u', var_name='a')     
delta = model.set_variable(var_type='_u', var_name='delta') 

# --- HARDCODED CONSTANT ---
l = 2.5

# Time-Varying Parameters
x_ref = model.set_variable(var_type='_tvp', var_name='x_ref')
y_ref = model.set_variable(var_type='_tvp', var_name='y_ref')

# Algebraic Slip Angle
beta = atan(0.5 * tan(delta))

# Right-Hand Side (RHS) differential equations
model.set_rhs('x_pos', v * cos(psi + beta))
model.set_rhs('y_pos', v * sin(psi + beta))
model.set_rhs('v',     a)
model.set_rhs('psi',   (v / l) * sin(beta))

model.setup()

# ==========================================
# 3. MPC CONTROLLER SETUP
# ==========================================
mpc = do_mpc.controller.MPC(model)

n_horizon = 5
setup_mpc = {
    'n_horizon': n_horizon,          
    't_step': 0.01,           
    'store_full_solution': True,
}
mpc.set_param(**setup_mpc)

# Objectives
mterm = (x_pos - x_ref)**2 + (y_pos - y_ref)**2  
lterm = (x_pos - x_ref)**2 + (y_pos - y_ref)**2  

mpc.set_objective(mterm=mterm, lterm=lterm)
mpc.set_rterm(a=0.1, delta=0.1)

# Hard Physical Boundaries
mpc.bounds['lower', '_u', 'a'] = -5.0      
mpc.bounds['lower', '_u', 'delta'] = -21  
mpc.bounds['lower', '_x', 'v'] = 0.0       

mpc.bounds['upper', '_u', 'a'] = 3.0       
mpc.bounds['upper', '_u', 'delta'] = 21   
mpc.bounds['upper', '_x', 'v'] = 25.0

# Register Initial Controller State
x0_start = np.array([42.0, 5.0, 0.0, np.pi/2])
mpc.x0 = x0_start 

# --- MPC Time-Varying Parameter (TVP) Function ---
tvp_template = mpc.get_tvp_template()

def tvp_fun(t_now):
    x_curr = float(mpc.x0['x_pos'])
    y_curr = float(mpc.x0['y_pos'])
    
    _, current_idx = track_tree.query([x_curr, y_curr])
    
    for k in range(n_horizon):
        target_idx = (current_idx + k) % len(x_mid)
        tvp_template['_tvp', k, 'x_ref'] = x_mid[target_idx]
        tvp_template['_tvp', k, 'y_ref'] = y_mid[target_idx]
        
    return tvp_template

mpc.set_tvp_fun(tvp_fun)

# Finalize MPC setup (No static parameter function needed!)
mpc.setup()

# ==========================================
# 4. SIMULATOR SETUP
# ==========================================
simulator = do_mpc.simulator.Simulator(model)
simulator.set_param(t_step=0.05)

# --- Simulator Time-Varying Parameter Function ---
sim_tvp_template = simulator.get_tvp_template()

def sim_tvp_fun(t_now):
    x_curr = float(simulator.x0['x_pos'])
    y_curr = float(simulator.x0['y_pos'])
    
    _, current_idx = track_tree.query([x_curr, y_curr])
    
    sim_tvp_template['x_ref'] = x_mid[current_idx]
    sim_tvp_template['y_ref'] = y_mid[current_idx]
    return sim_tvp_template

simulator.set_tvp_fun(sim_tvp_fun)
simulator.setup()

# Synchronize simulator state
simulator.x0 = x0_start
mpc.set_initial_guess()

# ==========================================
# 5. CLOSED-LOOP TIME STEPPING LOOP
# ==========================================
# 1200 iterations * 0.05 seconds = 60 seconds of driving
for step in range(1200):
    u0 = mpc.make_step(simulator.x0)
    simulator.make_step(u0)

# ==========================================
# 6. RETRIEVE LOG DETAILS AND PLOT
# ==========================================
x_history = mpc.data['_x', 'x_pos']
y_history = mpc.data['_x', 'y_pos']
v_history = mpc.data['_x', 'v']

fig = go.Figure()

# Plot Track Centerline
fig.add_trace(go.Scatter(
    x=x_mid, y=y_mid, 
    mode='lines', 
    name='Track Midline Reference', 
    line=dict(color='gray', dash='dash')
))

# Plot Vehicle Path
fig.add_trace(go.Scatter(
    x=x_history.flatten(), y=y_history.flatten(), 
    mode='lines+markers', 
    name='MPC Tracking Path',
    line=dict(color='mediumseagreen', width=3),
    marker=dict(size=3),
    text=[f"Speed: {v:.2f} m/s" for v in v_history.flatten()],
    hoverinfo='text+x+y'
))

fig.update_layout(
    title="Optimal Closed-Loop MPC Trajectory (Hardcoded Base)",
    xaxis_title="X Coordinate (meters)",
    yaxis_title="Y Coordinate (meters)",
    yaxis=dict(scaleanchor="x", scaleratio=1), 
    showlegend=True
)

fig.show()

In [ ]:
# ==========================================
# 4. SIMULATOR SETUP
# ==========================================
simulator = do_mpc.simulator.Simulator(model)
simulator.set_param(t_step=0.05)

# The simulator needs to read the exact same parameter function
simulator.set_p_fun(p_fun)
simulator.setup()

# Synchronize the simulator and estimator states
simulator.x0 = x0_start
mpc.set_initial_guess()

# ==========================================
# 5. OPEN-LOOP/CLOSED-LOOP TIME STEPPING
# ==========================================
# 1200 iterations * 0.05 seconds = 60 seconds of driving
for step in range(1200):
    # 1. Query MPC for optimized control adjustments [a, delta]
    u0 = mpc.make_step(simulator.x0)
    
    # 2. Feed commands to simulator to calculate new physical positions
    simulator.make_step(u0)

# ==========================================
# 6. RETRIEVE LOG DETAILS AND PLOT
# ==========================================
# Gather the computed state vectors from storage histories
x_history = mpc.data['_x', 'x_pos']
y_history = mpc.data['_x', 'y_pos']
v_history = mpc.data['_x', 'v']

fig = go.Figure()

# Plot CSV Racetrack Centerline
fig.add_trace(go.Scatter(
    x=x_mid, y=y_mid, 
    mode='lines', 
    name='Track Midline Reference', 
    line=dict(color='gray', dash='dash')
))

# Plot Actual Autonomous Vehicle Path
fig.add_trace(go.Scatter(
    x=x_history.flatten(), y=y_history.flatten(), 
    mode='lines+markers', 
    name='MPC Tracking Path',
    line=dict(color='mediumseagreen', width=3),
    marker=dict(size=3),
    text=[f"Speed: {v:.2f} m/s" for v in v_history.flatten()],
    hoverinfo='text+x+y'
))

fig.update_layout(
    title="Optimal Closed-Loop MPC Trajectory Output",
    xaxis_title="X Coordinate (meters)",
    yaxis_title="Y Coordinate (meters)",
    yaxis=dict(scaleanchor="x", scaleratio=1), # Retain authentic spatial dimensions
    showlegend=True
)

fig.show()